# Cloud Provider Analytics · Pipeline completo en Colab

Ejecuta la 2.ª entrega de punta a punta desde un entorno limpio:
Landing → Bronze (batch + streaming) → Silver → Gold → anomalías → AstraDB.

**Antes de empezar**, para la parte de AstraDB (sección 3):
1. En el panel izquierdo de Colab, ícono de llave (**Secrets**), crear el secreto
   `ASTRA_DB_TOKEN` con el token de AstraDB (empieza con `AstraCS:`) y habilitarle
   acceso a este notebook.
2. Tener a mano el **Secure Connect Bundle** (`secure-connect-*.zip`) para subirlo.

Las credenciales no se escriben en el notebook ni en el repo.

## 1. Entorno

In [ ]:
# Java 17 (requerido por Spark 4) y el repo. 1-2 minutos.
!apt-get update -qq > /dev/null
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

BRANCH = "main"   # versión entregada: tag entrega-2
!rm -rf /content/repo
!git clone -q --branch {BRANCH} https://github.com/santiagobolado/tp1-cloud-provider-analytics.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt
!java -version 2>&1 | head -1
!python -c "import pyspark; print('PySpark', pyspark.__version__)"

## 2. Pipeline

In [ ]:
# Reinicio limpio: las zonas derivadas y los checkpoints se regeneran desde Landing.
!rm -rf data/bronze data/silver data/gold data/quarantine _checkpoints evidence/runs

In [ ]:
!python -m src.ingest.batch_to_bronze          # 7 CSV -> Bronze

In [ ]:
!python -m src.ingest.stream_to_bronze         # 120 JSONL -> Bronze (Structured Streaming)

In [ ]:
!python -m src.transform.silver_job            # Bronze -> Silver

In [ ]:
!python -m src.transform.gold_stream           # flujo C: capa rápida, watermark 7 días

In [ ]:
!python -m src.transform.gold_job              # flujo D: reconciliación, mart oficial

In [ ]:
!python -m src.analytics.anomalies             # cost_anomaly_mart (MAD)

### Idempotencia: re-ejecutar no cambia los conteos

In [ ]:
!python -m src.ingest.batch_to_bronze 2>&1 | grep -E "leídas="
!python -m src.ingest.stream_to_bronze 2>&1 | grep -E "Esta corrida"
!python -m src.transform.silver_job 2>&1 | grep -E "Silver eventos"
!python -m src.transform.gold_job 2>&1 | grep -E "Gold org_daily"

In [ ]:
!python -m pytest tests -q

## 3. Serving en AstraDB

In [ ]:
# Subir el Secure Connect Bundle y leer el token desde Colab Secrets.
from google.colab import files, userdata
up = files.upload()
bundle = next(n for n in up if n.endswith(".zip"))
os.makedirs("/content/secrets", exist_ok=True)
os.replace(bundle, f"/content/secrets/{bundle}")
os.environ["ASTRA_DB_BUNDLE_PATH"] = f"/content/secrets/{bundle}"
os.environ["ASTRA_DB_TOKEN"] = userdata.get("ASTRA_DB_TOKEN")
os.environ["ASTRA_DB_KEYSPACE"] = "cloud_analytics"
print("Bundle:", os.environ["ASTRA_DB_BUNDLE_PATH"])

In [ ]:
# Crea las tablas, carga primero la capa rápida y después el mart reconciliado (upsert).
!python -m src.serving.load_cassandra --with-speed

In [ ]:
# Segunda carga: el COUNT(*) no cambia (idempotencia del serving).
!python -m src.serving.load_cassandra

In [ ]:
!python -m src.serving.run_queries
import glob
from IPython.display import Markdown, display
display(Markdown(open(sorted(glob.glob("evidence/serving/consultas_*.md"))[-1]).read()))

## 4. Evidencias

In [ ]:
# Descarga los logs de todas las corridas (Anexo A de la entrega).
!zip -qr /content/evidencias.zip evidence
files.download("/content/evidencias.zip")